In [4]:
# =====================================================================================
# 05_sqi_dev.ipynb  --  SQI gate development on DS1 ONLY (no DS2 data is read anywhere in this notebook)
# Needs src/sqi.py and src/noise.py (the file was called noice.py; rename it). Nothing is trained except the
# low-capacity SQI regression. The classifier artifact is only read. Outputs go to a NEW versioned directory.
# =====================================================================================

# ===== CELL 1: setup, primary artifact (read only), versioned output directory =====
from google.colab import drive
drive.mount('/content/drive')
!pip install -q wfdb
import os, sys, json, zlib, hashlib, platform, datetime, time
os.chdir('/content/drive/MyDrive/FYP'); sys.path.insert(0, os.getcwd())

import numpy as np, pandas as pd, scipy, matplotlib.pyplot as plt
from scipy.special import softmax
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score
from src import config as C
from src.data import load_beats, load_signal
from src.segment import bandpass
from src.noise import NOISE_TYPES, load_noise, CHUNK, BLOCK, chunk_starts
from src.calibration import fit_temperature
from src.selective import *
import src.sqi as Q
from src.sqi import FEATURES, beat_features, rule_score, FusedSQI, SNR_CLEAN_CAP

ARTIFACT = C.PROC / 'oof_ds1_v6_subjectfolds.npz'
ARTIFACT_SHA256 = '7138c8cf977be01aa1116ab78cab49da344413fd8ec5d8551846f39e37ab1e6c'
FOLDS = {0: [109, 118, 119, 115, 230], 1: [207, 106, 202, 201, 112, 122],
         2: [208, 124, 114, 116], 3: [203, 209, 101, 108], 4: [205, 215, 220, 223]}
EXPECTED_ENS_ACC = 0.782
SNRS = [24, 18, 12, 6, 0, -6]            # same grid as the planned DS2 evaluation
N_TARGET = 200                           # beats sampled per record and condition for the development table
COV_LEVELS = [0.95, 0.9, 0.8, 0.7]
R_WR, WR_SEED = 2000, 2031               # within-record null replicates (use 5000 for the final DS2 analysis)

sha256_of = lambda p: hashlib.sha256(open(p, 'rb').read()).hexdigest()
assert sha256_of(ARTIFACT) == ARTIFACT_SHA256, 'primary artifact hash differs'
k_ver = 1
while (C.PROC / f'sqi_dev_r{k_ver}').exists():
    k_ver += 1
OUT = C.PROC / f'sqi_dev_r{k_ver}'; OUT.mkdir(parents=True, exist_ok=False)
print('ARTIFACT:', ARTIFACT, '\nOUTPUT  :', OUT)

def save_df(name, df, index=False):
    p = OUT / name; assert not p.exists(), p; df.to_csv(p, index=index); return p

d = np.load(ARTIFACT)
L, y, rec, ridx, fold = d['logits'], d['y'], d['rec'], d['ridx'], d['fold']
n = len(y); assert L.shape == (3, n, 4)
rec2fold = {r: k for k, rs in FOLDS.items() for r in rs}
assert set(rec2fold) == set(C.DS1_PRIMARY) and rec2fold[201] == rec2fold[202]
assert np.array_equal(fold, np.array([rec2fold[int(r)] for r in rec])), 'fold layout differs from section 12'
Lens = L.mean(0); correct = Lens.argmax(1) == y
assert abs(correct.mean() - EXPECTED_ENS_ACC) < 0.002, 'ensemble accuracy differs from the primary baseline'
T = fit_temperature(Lens, y); conf_cal = softmax(Lens / T, 1).max(1)       # calibrated MSP, as in notebook 03
tbl = load_beats(); tbl = tbl[tbl.split == 'DS1'].sort_values(['record', 'r_idx'])
print('ensemble acc', correct.mean().round(3), '| T', round(T, 3), '| beats', n)


# ===== CELL 2: development noise pools (A for training, B for validation) and SNR validation =====
# Both pools are subsets of the 'dev' pool of src/noise.py (even 60 s blocks): A = blocks 0, 4, 8, ...; B = blocks 2, 6, 10, ...
# so training noise and validation noise never share a segment, and neither touches the 'test' pool (odd blocks) reserved for DS2.
_ps = {}
def pool_starts(kind, label):
    if (kind, label) not in _ps:
        nz = load_noise(kind); st = []
        for b in range(len(nz) // BLOCK):
            if b % 4 == (0 if label == 'A' else 2):
                st += list(range(b * BLOCK, (b + 1) * BLOCK - CHUNK + 1, CHUNK))
        _ps[(kind, label)] = np.array(st)
    return _ps[(kind, label)]

def noisy(rec_id, sig, kind, snr, label):
    """Same SNR definition as src.noise.add_noise (record-level signal variance), but drawing chunks from pool A or B."""
    rng = np.random.default_rng(zlib.crc32(f'sqi-{kind}-{snr}-{rec_id}-{label}'.encode()))
    nz = load_noise(kind); st = pool_starts(kind, label)
    z = np.concatenate([nz[s:s + CHUNK] for s in rng.choice(st, int(np.ceil(len(sig) / CHUNK)), replace=True)])[:len(sig)]
    z = z - z.mean()
    return (sig + np.sqrt(np.var(sig) / (np.var(z) * 10 ** (snr / 10))) * z).astype(np.float32)

sig106 = load_signal(106)
for kind in NOISE_TYPES:
    A, B = set(pool_starts(kind, 'A')), set(pool_starts(kind, 'B'))
    test = set(chunk_starts(len(load_noise(kind)), 'test'))
    assert not (A & B) and not ((A | B) & test), f'noise pools overlap for {kind}'
    for snr in [24, 6, -6]:
        for lab in ['A', 'B']:
            nsy = noisy(106, sig106, kind, snr, lab)
            got = 10 * np.log10(np.var(sig106) / np.var(nsy - sig106))
            assert abs(got - snr) < 0.5, (kind, snr, lab, got)
    print(kind, 'chunks A/B/test:', len(A), len(B), len(test), '| achieved SNR within 0.5 dB, pools disjoint: OK')


# ===== CELL 3: SQI features on DS1 under clean and development-noise conditions (sampled beats) =====
conds = [('clean', None, 'clean')] + [(k, s, lab) for k in NOISE_TYPES for s in SNRS for lab in ('A', 'B')]
rows, t0 = [], time.time()
for r in sorted(tbl.record.unique()):
    g = tbl[tbl.record == r]; r_all, yc = g.r_idx.values, g.aami.map(C.CLS2IDX).values
    rr = np.random.default_rng(zlib.crc32(f'sqi-targets-{r}'.encode()))
    tgt = np.sort(rr.choice(len(r_all), size=min(N_TARGET, len(r_all)), replace=False))   # same beats in every condition
    sig = load_signal(r)
    for kind, snr, lab in conds:
        s_n = sig if kind == 'clean' else noisy(r, sig, kind, snr, lab)
        F = beat_features(s_n, bandpass(s_n), r_all, tgt)
        rows.append(pd.DataFrame(F, columns=FEATURES).assign(
            record=r, fold=rec2fold[r], kind=kind, snr=float(SNR_CLEAN_CAP if kind == 'clean' else snr),
            label=lab, r_idx=r_all[tgt], y=yc[tgt]))
    print('record', r, 'done', round(time.time() - t0), 's')
dev = pd.concat(rows, ignore_index=True)
dev.to_csv(OUT / 'dev_features.csv.gz', index=False)
inv = dev.groupby('kind')[FEATURES].apply(lambda x: x.isna().mean()).round(4)
print('rows', len(dev), '| fraction of invalid (NaN) feature values by condition family:'); print(inv.to_string())


# ===== CELL 4: how each single SQI responds to the injected SNR (validation noise B + clean) =====
def fc(v):                                              # fail closed: NaN/-inf -> just below the smallest finite value
    v = np.asarray(v, float); fin = np.isfinite(v)
    return np.where(fin, v, (v[fin].min() - 1.0) if fin.any() else 0.0)

def auc_sp(df, score):
    s = np.asarray(score, float); snr = df.snr.values
    m = (snr <= 6) | (snr >= 18)
    auc = roc_auc_score(snr[m] >= 18, fc(s[m]))         # bad (<= 6 dB) versus good (>= 18 dB or clean)
    fin = np.isfinite(s)
    sp = spearmanr(s[fin], snr[fin])[0]
    return auc, sp

Brows = dev[(dev.label == 'B') | (dev.label == 'clean')]
res = []
for f in FEATURES:
    for kind in NOISE_TYPES:
        sub = Brows[(Brows.kind == 'clean') | (Brows.kind == kind)]
        a, s_ = auc_sp(sub, rule_score(sub[FEATURES].values, f))
        res.append([f, kind, a, s_])
single = pd.DataFrame(res, columns=['feature', 'noise', 'AUROC_bad_vs_good', 'Spearman_vs_SNR'])
print(single.pivot(index='feature', columns='noise', values='AUROC_bad_vs_good').round(3).assign(
    mean=lambda t: t.mean(1)).to_string())
print(single.pivot(index='feature', columns='noise', values='Spearman_vs_SNR').round(3).to_string())
save_df('single_feature_response.csv', single)
# PRE-DECLARED selection of the rule gate: the single SQI with the highest mean AUROC over the three noise types (DS1 validation noise only)
RULE_FEATURE = single.groupby('feature').AUROC_bad_vs_good.mean().idxmax()
print('RULE GATE FEATURE (selected on DS1 development data only):', RULE_FEATURE)


# ===== CELL 5: fused gate (ridge on SNR): subject-grouped CV, and leave-one-noise-type-out =====
A_rows = dev[(dev.label == 'A') | (dev.label == 'clean')]            # training noise
def fit_on(df): return FusedSQI().fit(df[FEATURES].values, df.snr.values)

parts = []
for k in range(5):                                                   # train on other folds' A noise, validate on this fold's B noise
    g = fit_on(A_rows[A_rows.fold != k]); va = Brows[Brows.fold == k]
    parts.append(va.assign(fused=g.score(va[FEATURES].values)))
cv = pd.concat(parts)
res = []
for kind in NOISE_TYPES:
    sub = cv[(cv.kind == 'clean') | (cv.kind == kind)]
    res.append(['fused (record-grouped CV)', kind, *auc_sp(sub, sub.fused.values)])
    res.append([f'rule: {RULE_FEATURE}', kind, *auc_sp(sub, rule_score(sub[FEATURES].values, RULE_FEATURE))])
for h in NOISE_TYPES:                                                # held-out noise TYPE (never seen in training)
    ps = []
    for k in range(5):
        tr = A_rows[(A_rows.fold != k) & (A_rows.kind != h)]
        g = fit_on(tr); va = Brows[(Brows.fold == k) & ((Brows.kind == 'clean') | (Brows.kind == h))]
        ps.append(va.assign(fused=g.score(va[FEATURES].values)))
    sub = pd.concat(ps); res.append(['fused (held-out noise type)', h, *auc_sp(sub, sub.fused.values)])
cvtab = pd.DataFrame(res, columns=['gate', 'noise', 'AUROC_bad_vs_good', 'Spearman_vs_SNR'])
print(cvtab.pivot(index='gate', columns='noise', values='AUROC_bad_vs_good').round(3).to_string())
print(cvtab.pivot(index='gate', columns='noise', values='Spearman_vs_SNR').round(3).to_string())
save_df('gate_validation.csv', cvtab)
gate_final = fit_on(dev)                                             # final fit on ALL DS1 development rows (clean + A + B)
print('final fused gate, standardised coefficients (dB per SD):',
      dict(zip(FEATURES, np.round(gate_final.coef, 3))), '| intercept', round(gate_final.intercept, 2))


# ===== CELL 6: clean DS1, matched coverage: confidence gate versus SQI gates (what each gate rejects when there is no added noise) =====
F_clean = np.full((n, len(FEATURES)), np.nan, np.float32)
for r in sorted(set(rec.tolist())):
    g = tbl[tbl.record == r]; r_all = g.r_idx.values
    pos = np.where(rec == r)[0]; tg = np.searchsorted(r_all, ridx[pos]); assert np.array_equal(r_all[tg], ridx[pos])
    sig = load_signal(r); F_clean[pos] = beat_features(sig, bandpass(sig), r_all, tg)
print('invalid (NaN) feature rows on clean DS1:', round(float(np.isnan(F_clean).any(1).mean()), 4))

sc = {'confidence (calibrated MSP)': conf_cal, f'SQI rule ({RULE_FEATURE})': rule_score(F_clean, RULE_FEATURE)}
sc_f = np.empty(n)
for k in range(5):
    gk = fit_on(A_rows[A_rows.fold != k]); m = fold == k; sc_f[m] = gk.score(F_clean[m])   # held-out fold only
sc['SQI fused (fold-held-out)'] = sc_f

idx_rec = {r: np.where(rec == r)[0] for r in np.unique(rec)}
cls_n = np.bincount(y, minlength=4).astype(float)
def wr_null(acc, R=R_WR, seed=WR_SEED):
    """Keep each record's accepted count, choose WHICH beats at random inside the record."""
    rng = np.random.default_rng(seed); k_rec = {r: int(acc[ix].sum()) for r, ix in idx_rec.items()}
    cov = np.array([acc[y == q].mean() for q in range(4)])
    nc = np.empty((R, 4))
    for i in range(R):
        cnt = np.zeros(4)
        for r, ix in idx_rec.items():
            if k_rec[r]: cnt += np.bincount(y[rng.choice(ix, k_rec[r], replace=False)], minlength=4)
        nc[i] = cnt / cls_n
    gap = cov[:3].max() - cov[:3].min(); ng = nc[:, :3].max(1) - nc[:, :3].min(1)
    return cov, nc.mean(0), gap, ng.mean(), (1 + (ng >= gap).sum()) / (R + 1), (1 + (ng <= gap).sum()) / (R + 1)

rows, rows_pc = [], []
for gname, s in sc.items():
    for c in COV_LEVELS:
        acc = accept_at_coverage(s, c)
        cov, ncov, gap, ngap, p_up, p_lo = wr_null(acc)
        rows.append([gname, f'{int(c*100)}%', selective_risk(acc, correct), *cov, gap, ngap, p_up, p_lo])
        for q, cl in enumerate(C.CLASSES):
            rows_pc.append([gname, f'{int(c*100)}%', cl, cov[q], ncov[q], cov[q] - ncov[q]])
cmp_tab = pd.DataFrame(rows, columns=['gate', 'coverage', 'selective_risk', 'cov_N', 'cov_S', 'cov_V', 'cov_F',
                                      'gap_NSV', 'within_record_null_gap', 'p_obs_larger', 'p_obs_smaller'])
pc_tab = pd.DataFrame(rows_pc, columns=['gate', 'coverage', 'class', 'observed_cov', 'within_record_null_cov', 'obs_minus_null'])
aurc_tab = pd.DataFrame([[g_, aurc(s, correct)] for g_, s in sc.items()], columns=['gate', 'AURC_clean_DS1'])
print(aurc_tab.round(4).to_string(index=False)); print(cmp_tab.round(4).to_string(index=False))
print(pc_tab[pc_tab['class'].isin(['S', 'V', 'F'])].round(4).to_string(index=False))
print('F is descriptive only. Within-record null: conditional on these 22 subjects and each gate\'s own per-record counts; exploratory.')
save_df('clean_ds1_gate_comparison.csv', cmp_tab); save_df('clean_ds1_per_class_deviation.csv', pc_tab); save_df('clean_ds1_aurc.csv', aurc_tab)

# 2 x 2 agreement at 80% coverage between the confidence gate and the fused SQI gate, by true class
a_conf = accept_at_coverage(sc['confidence (calibrated MSP)'], 0.8); a_sqi = accept_at_coverage(sc['SQI fused (fold-held-out)'], 0.8)
dis = pd.DataFrame({'class': np.array(C.CLASSES)[y], 'conf': np.where(a_conf, 'accept', 'reject'), 'sqi': np.where(a_sqi, 'accept', 'reject')})
agree = dis.groupby(['class', 'conf', 'sqi']).size().unstack(['conf', 'sqi'], fill_value=0)
print(agree.to_string()); save_df('clean_ds1_agreement_2x2.csv', agree.reset_index())


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ARTIFACT: /content/drive/MyDrive/FYP/data/processed/oof_ds1_v6_subjectfolds.npz 
OUTPUT  : /content/drive/MyDrive/FYP/data/processed/sqi_dev_r4
ensemble acc 0.782 | T 1.418 | beats 53138
bw chunks A/B/test: 48 42 90 | achieved SNR within 0.5 dB, pools disjoint: OK
em chunks A/B/test: 48 42 90 | achieved SNR within 0.5 dB, pools disjoint: OK
ma chunks A/B/test: 48 42 90 | achieved SNR within 0.5 dB, pools disjoint: OK
record 101 done 15 s
record 106 done 30 s
record 108 done 53 s
record 109 done 68 s
record 112 done 85 s
record 114 done 100 s


KeyboardInterrupt: 

In [ ]:


# ===== CELL 7: lock file (written only after the choices are recorded in the project log) =====
LOCK = True        # set True ONLY after the decisions below are written into the log; DS2 must not be touched before this
lock = dict(
    created=datetime.datetime.now().isoformat(timespec='seconds'), sqi_source_sha256=Q.source_hash(),
    features=FEATURES, window_samples=Q.WIN, fs=Q.FS, min_beats=Q.MIN_BEATS, flat_eps=Q.FLAT_EPS, snr_clean_cap=SNR_CLEAN_CAP,
    invalid_policy='NaN features -> score -inf (rejected first)',
    noise=dict(types=NOISE_TYPES, snrs=SNRS, training_pool='dev blocks 0,4,8,...', validation_pool='dev blocks 2,6,10,...',
               reserved_for_ds2='test pool (odd blocks), never read here'),
    rule_gate=dict(feature=RULE_FEATURE, selection='highest mean AUROC (bad <= 6 dB vs good >= 18 dB or clean) over noise types, DS1 validation noise'),
    fused_gate=gate_final.to_dict(),
    operating_thresholds_clean_ds1={g_: {f'{int(c*100)}%': float(np.sort(s)[::-1][int(round(c * n)) - 1]) for c in COV_LEVELS}
                                    for g_, s in sc.items() if g_.startswith('SQI')},
    artifact=str(ARTIFACT), artifact_sha256=ARTIFACT_SHA256, ds2_data_used=False,
    versions=dict(python=platform.python_version(), numpy=np.__version__, pandas=pd.__version__, scipy=scipy.__version__))
if LOCK:
    p = OUT / 'sqi_gate_lock.json'; assert not p.exists()
    json.dump(lock, open(p, 'w'), indent=2)
    print('LOCKED:', p, '| sha256', sha256_of(p))
else:
    print('NOT LOCKED (LOCK = False). Review cells 4-6 and record the gate choices in the log first.')


LOCKED: /content/drive/MyDrive/FYP/data/processed/sqi_dev_r3/sqi_gate_lock.json | sha256 34e589659cdeac9b0fb703f03df43bae289afb3759bd6d349e2e884dd1557518


In [7]:
##  SAMPLE EXPERIMENTS


import inspect
from src import selective

print(inspect.getsource(selective.accept_at_coverage))

def accept_at_coverage(score, coverage):
    n_keep = int(round(coverage * len(score)))
    mask = np.zeros(len(score), bool)
    mask[np.argsort(-score, kind='stable')[:n_keep]] = True
    return mask

